# 10000 selected Khmer OCR demonstration images
Run all, enter an example number from Gallery_10000.html, then click Run real prediction. CPU is sufficient. These images were selected after exact-match screening; this does not establish 100% accuracy on new images. The first screening subset used CPU and later screening used Apple MPS GPU. Numerical differences between runtimes are possible. Original source labels were not replaced with model predictions.

In [ ]:
%pip -q install torch pillow numpy huggingface_hub ipywidgets
from huggingface_hub import hf_hub_download
from pathlib import Path
import sys,zipfile,json
repo='Seoul12/khmer-ocr-three-architectures'
module=hf_hub_download(repo,'inference_three.py');sys.path.insert(0,str(Path(module).parent))
from inference_three import MultiRecognizer
archive=hf_hub_download('Seoul12/khmer-ocr-selected-demo','Khmer_OCR_10000_Selected.zip',repo_type='dataset')
base=Path('khmer_demo_10000');root=base/'printed'
with zipfile.ZipFile(archive) as z:z.extractall(base)
data=json.loads((root/'manifest.json').read_text());rows=data['images']
recognizer=MultiRecognizer(hf_hub_download(repo,'transformer.pt'))
print(len(rows),'selected exact-match printed images ready. Choose an example number.')


In [ ]:
import ipywidgets as widgets
from IPython.display import display,clear_output
from PIL import Image
number=widgets.BoundedIntText(value=1,min=1,max=len(rows),description='Example:')
run=widgets.Button(description='Run real prediction',button_style='primary')
output=widgets.Output()
def predict(_):
    with output:
        clear_output(wait=True)
        row=rows[number.value-1];path=root/row['image']
        display(Image.open(path));prediction=recognizer.predict(path)
        print('Example:',number.value,'Source validation row:',row['source_row_index'])
        print('ANSWER:',row['answer']);print('PREDICTION:',prediction)
        print('RIGHT — exact match' if prediction==row['answer'] else 'WRONG — review runtime or preprocessing')
        print('Selected demonstration example, not overall model accuracy.')
run.on_click(predict);display(number,run,output)


In [ ]:
TRY_YOUR_IMAGE=False
if TRY_YOUR_IMAGE:
    from google.colab import files
    for filename in files.upload():
        display(Image.open(filename));print('PREDICTION:',recognizer.predict(filename))
        print('No reference answer was provided; correctness is not automatically known.')
